# Notebook: Fine-tune COCO→Cityscapes (EoMT) – Colab-ready

Este cuaderno guía: 1) smoke-test del checkpoint COCO, 2) evaluación baseline en Cityscapes (mIoU), 3) fine-tuning (head-only / gradual unfreeze) y 4) opción LoRA.

## 1) Setup: repo, dependencias y GPU (ejecutar en Colab)

In [ ]:
%cd /content
# Clona el repo si estás en un entorno nuevo (Colab)
!git clone https://github.com/Escortiz/ML_RL.git || true
%cd /content/ML_RL/eomt
# Instala dependencias principales (usa el requirements del repo)
!pip install -r requirements.txt
# A veces conviene reinstalar PyTorch para la GPU de Colab (elige la URL correcta para tu runtime)
!pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio || true
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 || true
# Comprueba GPU
python -c "import torch; print('cuda:', torch.cuda.is_available(), 'device_count:', torch.cuda.device_count())"

## 2) Datos: instrucciones rápidas para Cityscapes
- Cityscapes requiere cuenta y descarga manual; sube los zips `leftImg8bit_trainvaltest.zip` y `gtFine_trainvaltest.zip` a `/content/ML_RL/cityscapes_dataset` en Colab (o monta Drive).
- Asegúrate de que la estructura sea la esperada por `eomt/datasets/cityscapes_semantic.py`.

In [ ]:
# Ejemplo: montar Google Drive (opcional)
from google.colab import drive
# drive.mount('/content/drive')  # descomenta si usas Drive
# Copia los zips a /content/ML_RL/cityscapes_dataset o ajusta --data.path en los comandos abajo

## 3) Smoke-test: cargar checkpoint COCO o Cityscapes y ejecutar una inferencia rápida
Este bloque adapta el código de `inference.ipynb` para cargar el config y el checkpoint y visualizar una muestra.

In [ ]:
import yaml, importlib, warnings
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from lightning import seed_everything
seed_everything(0, verbose=False)
device = 0 if torch.cuda.is_available() else 'cpu'
# Ajusta estas rutas según lo tengas en Colab
ACTIVE_YAML = '/content/ML_RL/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml'
ACTIVE_WEIGHTS = '/content/ML_RL/eomt/dataset_trained/eomt_coco.bin'  # cambia por el checkpoint COCO o Cityscapes
DATA_PATH = '/content/ML_RL/cityscapes_dataset'
# Cargar config de modelo (usa el config correcto)
with open(ACTIVE_YAML, 'r') as f:
    cfg = yaml.safe_load(f)
# Cargar DataModule (solo para obtener tamaños y clases)
dm_module_name, dm_class = cfg['data']['class_path'].rsplit('.', 1)
dm_cls = getattr(importlib.import_module(dm_module_name), dm_class)
dm_kwargs = cfg['data'].get('init_args', {})
dm = dm_cls(path=DATA_PATH, batch_size=1, num_workers=0, check_empty_targets=False, **dm_kwargs).setup()
# Build encoder + network + lit module similarly to inference.ipynb
warnings.filterwarnings('ignore', message=r
)
enc_cfg = cfg['model']['init_args']['network']['init_args']['encoder']
enc_mod, enc_cls = enc_cfg['class_path'].rsplit('.', 1)
Enc = getattr(importlib.import_module(enc_mod), enc_cls)
encoder = Enc(img_size=dm.img_size, **enc_cfg.get('init_args', {}))
net_cfg = cfg['model']['init_args']['network']
net_mod, net_cls = net_cfg['class_path'].rsplit('.', 1)
Net = getattr(importlib.import_module(net_mod), net_cls)
net_kwargs = {k: v for k, v in net_cfg.get('init_args', {}).items() if k != 'encoder'}
network = Net(masked_attn_enabled=False, num_classes=dm.num_classes, encoder=encoder, **net_kwargs)
lit_mod, lit_cls = cfg['model']['class_path'].rsplit('.', 1)
Lit = getattr(importlib.import_module(lit_mod), lit_cls)
model = Lit(img_size=dm.img_size, num_classes=dm.num_classes, network=network, **{k:v for k,v in cfg['model'].get('init_args', {}).items() if k!='network'}).eval().to(device)
# Load checkpoint weights (non-strict to allow mismatches)
state = torch.load(ACTIVE_WEIGHTS, map_location=f'cuda:{device}' if device!='cpu' else 'cpu', weights_only=False)
if 'state_dict' in state: state = state['state_dict']
model.load_state_dict(state, strict=False)
print('Checkpoint cargado. Ejecutando inferencia de muestra...')
# Infer y visualizar una imagen de validación
img_idx = 0
img, target = dm.val_dataloader().dataset[img_idx]
def apply_colormap(image):
    # simple normalization for visualization
    return image.transpose(1,2,0) if image.ndim==3 else image
with torch.no_grad(), autocast(dtype=torch.float16, device_type='cuda' if torch.cuda.is_available() else 'cpu'):
    imgs = [img.to(device)]
    crops, origins = model.window_imgs_semantic(imgs)
    mask_logits_per_layer, class_logits_per_layer = model(crops)
    mask_logits = F.interpolate(mask_logits_per_layer[-1], dm.img_size, mode='bilinear')
    crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
    logits = model.revert_window_logits_semantic(crop_logits, origins, [img.shape[-2:]])
    preds = logits[0].argmax(0).cpu().numpy()
    target_arr = model.to_per_pixel_targets_semantic([target], 255)[0].cpu().numpy()
# Mostrar
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(apply_colormap(img.cpu().numpy())); plt.title('Input'); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(preds); plt.title('Prediction'); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(target_arr); plt.title('Target'); plt.axis('off')
plt.show()

## 4) Evaluación baseline (mIoU) en Cityscapes
Usa el `validate` del CLI o el script `eval_iou.py` incluido. Ejecuta en la carpeta `eomt` del repo.

In [ ]:
# Ejemplo: ejecutar validación via CLI (ajusta rutas y dispositivos)
%cd /content/ML_RL/eomt
!python main.py validate \
  -c /content/ML_RL/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --trainer.devices 1 \
  --data.batch_size 2 \
  --data.path /content/ML_RL/cityscapes_dataset \
  --model.ckpt_path /content/ML_RL/eomt/dataset_trained/eomt_cityscapes.bin

## 5) Fine-tuning: comandos recomendados
- Fine-tune completo (start from COCO checkpoint): pasar `--model.ckpt_path` y opcionalmente `--model.load_ckpt_class_head False` si las clases difieren.
- Para ahorrar memoria: usar `--trainer.accumulate_grad_batches`, `--data.batch_size` pequeño y `--compile_disabled` si `torch.compile` da problemas.

In [ ]:
# Ejemplo: fine-tune desde checkpoint COCO (ajusta batch y devices para Colab)
%cd /content/ML_RL/eomt
!python main.py fit \
  -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --trainer.devices 1 \
  --data.batch_size 2 \
  --trainer.accumulate_grad_batches 8 \
  --data.path /content/ML_RL/cityscapes_dataset \
  --model.ckpt_path /content/ML_RL/eomt/dataset_trained/eomt_coco.bin \
  --model.load_ckpt_class_head False \
  --compile_disabled

### 5.a Head-only / freeze encoder (opción)
Si quieres entrenar solo la cabeza, hay dos caminos: 1) modificar el código para congelar `network.encoder` en el LightningModule, o 2) usar un pequeño wrapper que instancie el modelo, congele parámetros y lance un `Trainer` de Lightning.

In [ ]:
# Ejemplo ilustrativo: freeze encoder y ejecutar unas pocas épocas con Trainer (no usa la CLI).
# Puede requerir ajustes según la implementación interna del LightningModule.
from lightning.pytorch import Trainer
import yaml, importlib, torch
with open('/content/ML_RL/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml','r') as f: cfg=yaml.safe_load(f)
# Instancia modelos igual que en la celda de inferencia (omito detalles por brevedad)
# SUPONiendo que `lit_module` es la instancia del LightningModule:
# for p in lit_module.network.encoder.parameters():
#     p.requires_grad = False
# trainer = Trainer(devices=1, accelerator='gpu', max_epochs=3, precision='16-mixed')
# trainer.fit(lit_module, datamodule=dm)
print('Ejemplo de congelado: insértalo en tu flujo o pide ayuda para implementarlo en el repo si quieres que lo automatice.')

## 6) Gradual unfreeze (estrategia)
- Entrena la cabeza 3-5 épocas. Luego descongela las últimas N capas del encoder y continúa. Repite hasta descongelar todo.
- Registra mIoU tras cada etapa y compara con baseline.
- Usa learning rate menor para las capas descongeladas (por ejemplo, 0.1× la lr principal).

## 7) Opcional: LoRA (baja memoria) — guía rápida
- LoRA no está implementado nativamente en este repo. Para integrarlo, considera usar `loralib` o `peft` y adaptar las llamadas al `encoder` (aplicar adaptadores a los bloques de atención).
- Alternativa práctica: head-only → gradual unfreeze; implementar LoRA solo si necesitas mejor eficiencia y estás cómodo editando `models/eomt.py` para inyectar adapters.

## 8) Guardar resultados y comparar mIoU
- Tras cada experimento, ejecuta `!python eval_iou.py --datadir /content/ML_RL/cityscapes_dataset --subset val` para obtener mIoU.
- Guarda las métricas en un CSV (wandb o local) y compara la línea base (checkpoint proporcionado) frente a tu modelo fine-tuned.